In [ ]:
import sys
import os


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


sys.path.append(os.path.abspath(".."))
from package.SingleCharTokenizer import * 
from package.TextDataset import * 
from package.EmbeddingLayer import * 
from package.AttentionLayer import * 
from package.TransformerLayer import * 
from package.GPT import * 

%load_ext autoreload
%autoreload 2

In [2]:
d_emb = 16
context_window = 32
batch_size = 4

nb_heads = 4
d_emb_in = 32
d_k = 16
d_emb_out = 64

nb_layers = 10

In [3]:
tokenizer = SingleCharTokenizer()
tokens = torch.tensor(tokenizer.load_tokens("tokens.tok"))



vocab_size = tokenizer.vocab_size
print("vocab_size : " , vocab_size)

dataset = TextDataset(tokens,context_window=context_window)


loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

X,Y=next(iter(loader))
print(X.dtype)

vocab_size :  65
torch.int64


In [4]:
x = torch.rand((batch_size,context_window,d_emb_in),dtype=torch.float)

In [5]:

layer_embed = EmbeddingLayer(vocab_size, context_window, d_emb)


layer_mono = SelfAttention(d_emb_in=d_emb_in,
                           d_k=d_k,
                           d_emb_out=d_emb_out)


layer_multi = MultiHeadAttention(d_emb_in=d_emb_in,
                           d_k=d_k,
                           d_emb_out=d_emb_out,
                           nb_heads=nb_heads)

layer_ff = FeedForwardLayer(d_emb_in=d_emb_in,
                           d_k=d_k,
                           d_emb_out=d_emb_out)

gpt = GPT(vocab_size=vocab_size, context_window=context_window, d_emb=d_emb,nb_layers=nb_layers,nb_heads=nb_heads)

In [6]:
print(layer_embed.forward(X).shape)
print(layer_mono.forward(x).shape)
print(layer_multi.forward(x).shape)
print(layer_ff.forward(x).shape)

print(gpt.forward(X).shape)

torch.Size([4, 32, 16])
torch.Size([4, 32, 64])
torch.Size([4, 32, 64])
torch.Size([4, 32, 64])
torch.Size([4, 32, 65])


In [7]:
gpt.forward(X)

tensor([[[-1.0190, -0.1896,  1.2126,  ..., -0.5148,  0.7953,  0.1231],
         [-0.6536,  0.1466,  1.2041,  ...,  0.6476,  0.3131,  0.0613],
         [ 0.0492,  0.1021,  0.7804,  ..., -0.2337, -0.6554,  0.3201],
         ...,
         [ 0.4757, -0.2961,  0.8630,  ..., -0.9657, -0.4482,  0.1396],
         [-0.3174, -1.0354, -0.1156,  ...,  0.2388,  0.6365, -0.0498],
         [ 0.2121,  0.0395, -0.0450,  ...,  0.6636,  0.3798, -0.6929]],

        [[-0.8343,  0.1357,  1.5117,  ..., -0.1589,  0.1390,  0.0490],
         [-0.7251,  1.0705,  0.3413,  ...,  0.8259, -0.0485,  0.5221],
         [-0.2776, -0.0604,  0.6558,  ...,  0.2628, -1.1368,  0.6713],
         ...,
         [ 0.0314, -0.7535,  1.1025,  ..., -0.0258, -0.8134, -0.1807],
         [-0.4539, -0.5593, -0.2463,  ..., -0.5213,  0.9554, -0.1272],
         [-0.3231,  0.2877, -0.2315,  ..., -0.0657,  1.3091, -0.3208]],

        [[-0.8343,  0.1357,  1.5117,  ..., -0.1589,  0.1390,  0.0490],
         [-0.4193,  1.0585,  0.1748,  ...,  0

In [20]:
batch_size = 16
context_windows = 128
d_emb = 32


x = torch.rand((batch_size,context_windows,d_emb))

In [22]:
batch_size = 3
context_windows = 16
nb_heads = 4
d_emb_in = 8
d_k = 3
d_emb_out = 12


X = torch.arange(0,batch_size*context_windows*nb_heads*d_k)

X = X.view(batch_size,context_windows,nb_heads*d_k)

print(X.view(batch_size,context_windows,nb_heads,d_k).transpose(1,2))

tensor([[[[  0,   1,   2],
          [ 12,  13,  14],
          [ 24,  25,  26],
          [ 36,  37,  38],
          [ 48,  49,  50],
          [ 60,  61,  62],
          [ 72,  73,  74],
          [ 84,  85,  86],
          [ 96,  97,  98],
          [108, 109, 110],
          [120, 121, 122],
          [132, 133, 134],
          [144, 145, 146],
          [156, 157, 158],
          [168, 169, 170],
          [180, 181, 182]],

         [[  3,   4,   5],
          [ 15,  16,  17],
          [ 27,  28,  29],
          [ 39,  40,  41],
          [ 51,  52,  53],
          [ 63,  64,  65],
          [ 75,  76,  77],
          [ 87,  88,  89],
          [ 99, 100, 101],
          [111, 112, 113],
          [123, 124, 125],
          [135, 136, 137],
          [147, 148, 149],
          [159, 160, 161],
          [171, 172, 173],
          [183, 184, 185]],

         [[  6,   7,   8],
          [ 18,  19,  20],
          [ 30,  31,  32],
          [ 42,  43,  44],
          [ 54,  55,  56

In [16]:
class testNorm(nn.Module):
    def __init__(self, d_emb):
        super().__init__()
        

        self.d_emb = d_emb

        self.norm = nn.LayerNorm(d_emb)


    def forward(self, X):
        batch_size, context_window, d_emb = X.shape
        # [ batch_size , context_window , d_emb_in ]
        return self.norm(X)
        







In [29]:
batch_size = 3
context_windows = 12
d_emb = 4


X = torch.arange(0,batch_size*context_windows*d_emb,dtype=torch.float)
X = X.view(batch_size,context_windows,d_emb)

print(X.shape)


norm = testNorm(d_emb)

torch.Size([3, 12, 4])


In [27]:
norm.forward(X)

tensor([[[-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416]],

        [[-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416],
         [-

In [23]:
batch_size = 16
context_windows = 128

nb_heads = 4
d_emb_tok = 32


In [30]:
X = torch.arange(0,batch_size*context_windows*d_emb_tok,dtype=torch.float)


X = X.view(batch_size,context_windows,d_emb_tok)



X = torch.rand((batch_size,context_windows,d_emb_tok),dtype=torch.float)

In [31]:
layer = TransformerLayer(d_emb_tok=d_emb_tok,
                         nb_heads=nb_heads)

In [32]:
layer.forward(X)

tensor([[[ 5.5766e-01, -1.6414e-01,  7.8839e-01,  ...,  6.0972e-01,
           4.1109e-01,  1.7361e-02],
         [ 2.6587e-01,  2.3542e-01,  3.2775e-01,  ...,  7.0970e-01,
           3.2105e-01,  4.4763e-03],
         [ 9.0730e-01,  2.7396e-01,  1.0321e-01,  ...,  4.8567e-01,
           7.8078e-01,  2.5602e-01],
         ...,
         [ 2.3870e-02,  6.0950e-01,  1.3183e+00,  ...,  6.5783e-02,
           4.7021e-01,  2.0321e-01],
         [-1.2096e-01, -1.0968e-01,  7.0365e-01,  ...,  1.8472e-01,
           7.7899e-01, -1.1133e-01],
         [ 6.1754e-01,  7.1582e-02, -1.9122e-02,  ...,  1.2817e-01,
           1.0769e+00,  7.5151e-01]],

        [[-7.6713e-02,  1.5180e+00,  5.8079e-01,  ...,  6.7202e-01,
           4.3672e-01,  1.5502e+00],
         [ 3.8042e-01,  3.6018e-01,  1.1670e-01,  ...,  2.7161e-01,
           7.4218e-01,  4.3536e-01],
         [ 1.2013e-03,  7.8147e-01,  1.2057e+00,  ..., -4.6100e-01,
           2.9234e-01,  7.0838e-01],
         ...,
         [ 3.5191e-01,  8

In [33]:


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=6, d_ff=1024, block_size=128):
        super().__init__()
        
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerLayer(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

        self.block_size = block_size

    def forward(self, idx):
        B, T = idx.shape
        token_embeddings = self.token_emb(idx)           # [B, T, d_model]
        positions = torch.arange(T, device=idx.device)
        pos_embeddings = self.pos_emb(positions)[None, :, :]
        x = token_embeddings + pos_embeddings

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)  # [B, T, vocab_size]
        return logits

In [9]:
from package.SingleCharTokenizer import * 
from package.TextDataset import * 
from torch.utils.data import DataLoader

tokenizer = SingleCharTokenizer()
tokens = torch.tensor(tokenizer.load_tokens("tokens.tok"), dtype=torch.long) # check type



d_emb = 3
context_windows = 32
batch_size = 4
vocab_size = tokenizer.vocab_size
print("vocab_size : " , vocab_size)

dataset = TextDataset(tokens,context_window=context_windows)


loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

X,Y=next(iter(loader))

print(tokenizer.decode(X[0].tolist()))
print(tokenizer.decode(Y[0].tolist()))

print(X.shape)

vocab_size :  65
er celle qu'il avait perdue. le 
r celle qu'il avait perdue. le g
torch.Size([4, 32])


In [5]:
E = nn.Embedding(num_embeddings=vocab_size,embedding_dim=d_emb)
P = nn.Embedding(num_embeddings=context_windows,embedding_dim=d_emb)

In [6]:
i = 0
#x = tokens[i:i+context_windows]
emb_vocab = E(X)
print(X.shape)
print(emb_vocab.shape)

torch.Size([4, 32])
torch.Size([4, 32, 3])


In [7]:
position = torch.arange(0,context_windows)
print(position)

emb_pos = P(position)
print(emb_pos.shape)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])
torch.Size([32, 3])


In [8]:
(emb_vocab + emb_pos).shape

torch.Size([4, 32, 3])

In [ ]:
voc = torch.zeros((batch_size,context_windows,d_emb))
pos = torch.ones((context_windows,d_emb))

pos = torch.arange(context_windows)

emb_pos = P(position)
print(emb_pos.shape)

torch.Size([32, 3])


In [14]:
emb = (voc + emb_pos)
print(emb)

tensor([[[-0.1332,  0.7169, -0.9260],
         [-0.1700, -0.7339,  0.6290],
         [-0.5271, -0.0920, -0.0203],
         [ 0.8346, -0.4291,  0.2561],
         [-0.3879,  1.5616,  0.1393],
         [-1.0471,  0.4041,  0.3464],
         [-1.1617,  0.6215, -0.0955],
         [ 0.0635,  0.0968,  0.0425],
         [-0.5372,  1.5676,  1.1374],
         [-0.5702,  1.3451, -0.1401],
         [ 0.7720,  0.4197, -0.1769],
         [-1.1151, -0.5680, -0.8497],
         [ 1.0682,  0.6622, -0.7953],
         [ 0.0139,  0.1565, -1.9850],
         [-0.1317,  0.8892, -0.3485],
         [-0.3408, -0.3320, -0.8908],
         [ 0.3428,  1.6691, -0.5664],
         [-0.3950,  0.9587,  1.3574],
         [-0.0940,  1.3315, -2.1391],
         [ 2.2247,  2.6384,  0.3043],
         [-0.8864,  1.6258, -1.4421],
         [ 0.3968,  0.1045,  1.5242],
         [-0.3360, -0.3659, -0.6658],
         [-0.1385, -2.8633, -0.9359],
         [-0.7050,  0.4967, -0.6817],
         [-0.6549, -0.4678,  0.2307],
         [-0